# TAMP-OS Image Check — will my image work?

Before printing, this notebook tells you **whether an image will make a good tactile lithograph**, what will go wrong, and **how to fix it**: with a TAMP-OS setting (blur, resolution, print width, invert) or with a quick step in ImageJ/Fiji (crop, subtract background, split channels).

It uses the **same image processing as `tamp_batch_gui_v2.ipynb`** (grayscale → resize → contrast stretch → blur → invert) and then approximates what the printer can reproduce (nozzle width, layer height).

**How to use it**
1. Run Steps 1–3 once (install, imports, functions).
2. In Step 4, set `IMAGE_PATH` to your image and run the cell. You get a report and a preview.
3. If the report suggests a fix, Step 5 shows how to apply it and save a fixed image for the GUI.

> ⚠️ **Honest limits.** TAMP-OS has not been tested with users yet. "Touchable" here is a *proxy* based on braille dot dimensions (dot ~1.5 mm wide, ~0.5 mm high), not a user study. The print simulation is approximate. Use the check to catch problems early, and `tamp_resolution_compare.ipynb` plus a test print to confirm.

> Runs in Jupyter Lab, Jupyter Notebook or VS Code (no GUI window needed). Open it from the `image_check/` folder so the example paths work. More details in [`README.md`](README.md).

### Step 1 — Install dependencies (once)

In [ ]:
!pip install numpy pillow scipy matplotlib

### Step 2 — Imports

In [ ]:
# ── TAMP-OS image check: core functions ─────────────────────────────────────
# Self-contained: numpy, Pillow, scipy, matplotlib (same stack as the GUI notebook).
import math
import numpy as np
from PIL import Image
from scipy.ndimage import gaussian_filter, convolve, label, find_objects, grey_opening
from pathlib import Path
%matplotlib inline
print("Imports OK")

### Step 3 — Functions *(self-contained — no external files needed)*

Run all the cells in this step. You don't need to read them.

**Settings.** Change `PRINT_WIDTH_MM`, `NOZZLE_MM` and `LAYER_MM` to match your printer. The touch thresholds come from braille dot dimensions (see the note above).

In [ ]:
# 1. Settings
# Tactile proxy thresholds. TAMP-OS has not been tested with users yet, so these
# are anchored to standard braille dot dimensions (dot base ~1.5 mm, dot height
# ~0.5 mm, dot spacing ~2.3-2.5 mm): if a structure is smaller or lower than a
# braille dot, we flag it as "probably hard to feel".
MIN_FEATURE_MM = 1.5   # smallest structure diameter considered touchable
MIN_RELIEF_MM  = 0.5   # smallest height step considered touchable
SNR_OK         = 4.0   # printed structure height / printed noise roughness
SNR_MIN        = 2.0   # below this the structure is buried in roughness
# Printed noise roughness above one layer height is grain the printer would not
# otherwise produce - it reads as sandpaper on top of the structure.

PRINT_WIDTH_MM = 100.0
NOZZLE_MM      = 0.4
LAYER_MM       = 0.12
RELIEF_MM      = 3.0

# Same preset logic as tamp_batch_gui_v2.ipynb (calculate_presets)
def calculate_presets(print_size_x=PRINT_WIDTH_MM, nozzle_mm=NOZZLE_MM):
    def snap(v): return max(64, min(512, int(2 ** round(math.log2(v)))))
    return {
        "Low":    {"resolution": snap(print_size_x / (nozzle_mm * 2)),   "blur": 2.0},
        "Medium": {"resolution": snap(print_size_x / nozzle_mm),         "blur": 1.2},
        "High":   {"resolution": min(512, snap(print_size_x / (nozzle_mm * 0.5))), "blur": 0.8},
    }

**The TAMP-OS pipeline** — identical maths to `image_to_heightmap` in the GUI, plus an approximate print simulation.

In [ ]:
# 2. Pipeline
def to_pil(img):
    """Accept a path, PIL image or numpy array; return a PIL image."""
    if isinstance(img, Image.Image):
        return img
    if isinstance(img, np.ndarray):
        a = img
        if a.dtype != np.uint8:
            a = np.clip(a, 0, 255).astype(np.uint8)
        return Image.fromarray(a)
    return Image.open(img)


def resample_gray(img, resolution):
    """Grayscale + LANCZOS resize to `resolution` px wide (aspect preserved)."""
    img = to_pil(img).convert("L")
    w, h = img.size
    img = img.resize((resolution, max(2, round(resolution * h / w))), Image.LANCZOS)
    return np.array(img, dtype=np.float32)


def tamp_heightmap(img, resolution=256, blur=1.2, invert=False,
                   contrast_percentile=(2.0, 98.0)):
    """Normalized height map in [0, 1], exactly as TAMP-OS builds it (no flip)."""
    arr = resample_gray(img, resolution)
    lo, hi = np.percentile(arr, contrast_percentile)
    arr = np.clip((arr - lo) / (hi - lo + 1e-8), 0.0, 1.0)
    if blur > 0:
        arr = gaussian_filter(arr, sigma=blur)
    if invert:
        arr = 1.0 - arr
    return arr


def simulate_print(hm, print_width_mm=PRINT_WIDTH_MM, relief_mm=RELIEF_MM,
                   nozzle_mm=NOZZLE_MM, layer_mm=LAYER_MM):
    """Approximate what the printer can reproduce: the extrusion acts like a
    blur with FWHM = nozzle width, and heights are quantized to the layer
    height. Returns relief in mm (base excluded)."""
    px_mm = print_width_mm / hm.shape[1]
    z = hm * relief_mm
    z = gaussian_filter(z, sigma=(nozzle_mm / 2.355) / px_mm)
    return np.round(z / layer_mm) * layer_mm

**Image measurements**

| Measurement | What it means |
|---|---|
| structure size (mm on the print) | dominant size of the structures, from the image's spatial frequencies |
| contrast-to-noise | how strong the structure is compared with pixel noise *after* TAMP-OS resizes the image |
| colour gain | how much more contrast one colour channel / stain has than the grayscale TAMP-OS uses |
| brightness trend | how much of the image is a smooth gradient (uneven illumination, vignetting) |
| annotations | flat, saturated rectangles: scale bars, data bars, labels |
| background level | near-white background → invert ON (brightfield / histology) |

In [ ]:
# 3. Measurements
_IMMERKAER = np.array([[1, -2, 1], [-2, 4, -2], [1, -2, 1]], dtype=np.float32)

def noise_sigma(arr):
    """Fast noise std estimate (Immerkaer 1996). Fine texture counts as noise
    too, so treat it as 'pixel-scale roughness'."""
    a = arr.astype(np.float32)
    r = np.abs(convolve(a, _IMMERKAER, mode="reflect"))[1:-1, 1:-1]
    return float(math.sqrt(math.pi / 2) * r.mean() / 6.0)


def cnr_print(img, resolution):
    """Contrast-to-noise ratio *after* TAMP-OS resampling (what the print sees):
    full structure range (p2-p98) / pixel noise."""
    arr = resample_gray(img, resolution)
    smooth = gaussian_filter(arr, 1.0)
    rng = float(np.subtract(*np.percentile(smooth, [98, 2])))
    return rng / max(noise_sigma(arr), 1e-3)


def feature_size_px(gray):
    """Dominant structure diameter in source pixels, from the radially averaged
    power spectrum (noise floor removed). Returns (diameter_px, spectrum)."""
    a = gray.astype(np.float32)
    a = a - a.mean()
    h, w = a.shape
    win = np.outer(np.hanning(h), np.hanning(w))
    F = np.abs(np.fft.fftshift(np.fft.fft2(a * win))) ** 2
    fy = np.fft.fftshift(np.fft.fftfreq(h))[:, None]
    fx = np.fft.fftshift(np.fft.fftfreq(w))[None, :]
    k = np.hypot(fy, fx)                             # cycles / pixel
    bins = np.geomspace(1.0 / max(h, w) * 2, 0.5, 40)
    idx = np.digitize(k.ravel(), bins)
    P = np.array([F.ravel()[idx == i].mean() if np.any(idx == i) else 0
                  for i in range(1, len(bins))])
    kc = np.sqrt(bins[1:] * bins[:-1])
    floor = np.median(P[kc > 0.35]) if np.any(kc > 0.35) else 0
    # keep only bands clearly above the white-noise floor
    Pn = np.where(P > 2.0 * floor, P - floor, 0.0)
    # energy per log-frequency band: k^2 * P(k) peaks at the dominant scale
    E = Pn * kc ** 2
    i = int(np.argmax(E))
    # a structure of diameter d (spaced ~2.5 d) puts its energy near k ~ 1/(2.5 d)
    return 1.0 / (2.5 * kc[i]), (kc, E)


def touchable_fraction(gray, print_width_mm=PRINT_WIDTH_MM, min_mm=MIN_FEATURE_MM):
    """Share of the structural detail (noise removed) at sizes ≥ min_mm on the
    print."""
    kc, E = feature_size_px(gray)[1]
    d_mm = 1.0 / (2.5 * kc) * print_width_mm / gray.shape[1]
    tot = E.sum()
    return float(E[d_mm >= min_mm].sum() / tot) if tot > 0 else 1.0


def _structure_cnr(c):
    """Contrast-to-noise of one channel after removing the large-scale trend."""
    c = c - gaussian_filter(c, c.shape[1] / 20)
    sm = gaussian_filter(c, 1.5)
    return float(np.subtract(*np.percentile(sm, [98, 2]))) / max(noise_sigma(c), 1e-3)


def colour_gain(img):
    """How much more contrast-to-noise a single colour channel or stain has than
    the grayscale TAMP-OS uses. Returns (gain, best_name)."""
    rgb = np.array(to_pil(img).convert("RGB"), dtype=np.float32)
    L = np.array(to_pil(img).convert("L"), dtype=np.float32)
    H, D = rgb_to_hed(rgb)
    cands = {"red": rgb[..., 0], "green": rgb[..., 1], "blue": rgb[..., 2],
             "hematoxylin": H, "DAB": D}
    sL = max(_structure_cnr(L), 1e-3)
    scores = {n: _structure_cnr(c) for n, c in cands.items()}
    best = max(scores, key=scores.get)
    return scores[best] / sL, best


def rgb_to_hed(rgb):
    """Colour deconvolution for H-DAB / H&E (Ruifrok & Johnston 2001).
    Returns hematoxylin and DAB/eosin optical densities, scaled so that
    more stain = brighter."""
    od = -np.log(np.clip(rgb, 1, 255) / 255.0)
    M = np.array([[0.65, 0.70, 0.29],
                  [0.27, 0.57, 0.78],
                  [0.0, 0.0, 0.0]])
    M[2] = np.cross(M[0], M[1])
    M = M / np.linalg.norm(M, axis=1, keepdims=True)
    st = od.reshape(-1, 3) @ np.linalg.inv(M)
    st = st.reshape(rgb.shape)
    return st[..., 0], st[..., 1]


def illumination_trend(gray):
    """R² of a 2nd-order polynomial surface fit: high = brightness changes
    smoothly across the frame (uneven illumination, vignetting) - or one very
    large object, so check visually."""
    g = np.asarray(gray, np.float32)
    small = Image.fromarray(g).resize((128, max(2, round(128 * g.shape[0] / g.shape[1]))))
    a = np.array(small, np.float32)
    h, w = a.shape
    yy, xx = np.mgrid[0:h, 0:w]
    x = xx.ravel() / w - 0.5; y = yy.ravel() / h - 0.5
    A = np.stack([np.ones_like(x), x, y, x * x, y * y, x * y], 1)
    coef, *_ = np.linalg.lstsq(A, a.ravel(), rcond=None)
    fit = A @ coef
    return float(1 - np.var(a.ravel() - fit) / (np.var(a.ravel()) + 1e-8))


def find_annotations(gray, min_area_frac=0.001):
    """Flat, saturated, rectangular regions = scale bars, data bars, labels.
    Returns list of bounding boxes (x0, y0, x1, y1) in source pixels."""
    g = gray.astype(np.float32)
    flat = gaussian_filter(np.abs(g - gaussian_filter(g, 2)), 2) < 1.5
    sat = (g >= 250) | (g <= 5)
    from scipy.ndimage import binary_closing
    mask = binary_closing(sat & flat, structure=np.ones((9, 9)))
    lab, n = label(mask)
    boxes = []
    if n == 0:
        return boxes
    areas = np.bincount(lab.ravel())
    for i, sl in enumerate(find_objects(lab), start=1):
        if sl is None or areas[i] < min_area_frac * g.size:
            continue
        hgt = sl[0].stop - sl[0].start; wid = sl[1].stop - sl[1].start
        fill = areas[i] / (hgt * wid)
        if fill > 0.6 and min(hgt, wid) >= 4:
            boxes.append((sl[1].start, sl[0].start, sl[1].stop, sl[0].stop))
    # merge boxes that touch or nearly touch (a label split by its own text)
    merged = True
    while merged and len(boxes) > 1:
        merged = False
        for i in range(len(boxes)):
            for j in range(i + 1, len(boxes)):
                a, b = boxes[i], boxes[j]
                near = not (a[2] + 15 < b[0] or b[2] + 15 < a[0] or a[3] + 15 < b[1] or b[3] + 15 < a[1])
                hh = max(a[3] - a[1], b[3] - b[1])
                vov = min(a[3], b[3]) - max(a[1], b[1])          # same text line?
                gap = max(a[0], b[0]) - min(a[2], b[2])
                if near or (vov > 0.5 * min(a[3] - a[1], b[3] - b[1]) and gap < 3 * hh):
                    boxes[i] = (min(a[0], b[0]), min(a[1], b[1]), max(a[2], b[2]), max(a[3], b[3]))
                    boxes.pop(j); merged = True
                    break
            if merged:
                break
    return boxes


def suggest_invert(gray, is_colour=False):
    """Poster rule: bright background -> invert ON so structures print raised.
    Typical case: brightfield / histology (stained tissue on a near-white
    background). Background = median of a border band.
    Brightness alone cannot always separate structure from background (SEM
    shadows and edge glow, phase images), so this is a suggestion: the question
    to ask is 'are the structures I want to feel darker than the background?'.
    Returns (suggest_invert: bool, background_level: 0-255)."""
    g = gray.astype(np.float32)
    b = max(2, min(g.shape) // 20)
    border = np.concatenate([g[:b].ravel(), g[-b:].ravel(), g[:, :b].ravel(), g[:, -b:].ravel()])
    bg = float(np.median(border))
    return bool(bg >= 220 or (is_colour and bg >= 185)), bg

**Preparation helpers**: the Python versions of the Fiji fixes, so you can fix and save the image here if you prefer.

In [ ]:
# 4. Preparation helpers
def crop(img, box):
    """box = (x0, y0, x1, y1) in pixels."""
    return to_pil(img).crop(box)


def crop_out_annotations(img, boxes, margin=8):
    """Largest rectangle that avoids every annotation box (plus a safety margin
    for outlines and anti-aliased edges)."""
    im = to_pil(img); w, h = im.size
    boxes = [(max(0, x0 - margin), max(0, y0 - margin), min(w, x1 + margin), min(h, y1 + margin))
             for x0, y0, x1, y1 in boxes]
    best = (0, 0, w, h); best_area = 0
    tops = [0] + [b[3] for b in boxes]; bots = [h] + [b[1] for b in boxes]
    lefts = [0] + [b[2] for b in boxes]; rights = [w] + [b[0] for b in boxes]
    for y0 in tops:
        for y1 in bots:
            for x0 in lefts:
                for x1 in rights:
                    if y1 - y0 < 8 or x1 - x0 < 8:
                        continue
                    if any(not (bx1 <= x0 or bx0 >= x1 or by1 <= y0 or by0 >= y1)
                           for bx0, by0, bx1, by1 in boxes):
                        continue
                    a = (x1 - x0) * (y1 - y0)
                    if a > best_area:
                        best, best_area = (x0, y0, x1, y1), a
    return im.crop(best), best


def subtract_background(img, radius_px):
    """Rough equivalent of Fiji 'Process > Subtract Background' (rolling ball):
    grey opening with a square of side 2*radius, then smoothed."""
    g = np.array(to_pil(img).convert("L"), np.float32)
    bg = gaussian_filter(grey_opening(g, size=(2 * radius_px + 1,) * 2), radius_px / 2)
    out = g - bg
    out = 255 * (out - out.min()) / (np.ptp(out) + 1e-8)
    return Image.fromarray(out.astype(np.uint8))


def pick_channel(img, name):
    """'red' | 'green' | 'blue' | 'hematoxylin' | 'DAB' -> grayscale PIL image."""
    rgb = np.array(to_pil(img).convert("RGB"), np.float32)
    if name in ("red", "green", "blue"):
        c = rgb[..., "rgb".index(name[0])]
    else:
        H, D = rgb_to_hed(rgb)
        c = H if name == "hematoxylin" else D
    c = 255 * (c - c.min()) / (np.ptp(c) + 1e-8)
    return Image.fromarray(c.astype(np.uint8))


def median_denoise(img, size=3):
    from scipy.ndimage import median_filter
    g = np.array(to_pil(img).convert("L"))
    return Image.fromarray(median_filter(g, size=size))

**Synthetic test patterns + the reference map**: disks of known size and noise run through the pipeline, then scored.

In [ ]:
# 5. Synthetic patterns + map
def synth_bumps(d_mm, noise_sd, print_width_mm=PRINT_WIDTH_MM, src_px=1024,
                bg=80.0, amp=100.0, seed=0):
    """Disks of diameter d_mm (on the print), centre spacing 2.5*d, plus white
    noise. Returns (uint8 image, centre-distance map in mm)."""
    rng = np.random.default_rng(seed)
    mm_per_px = print_width_mm / src_px
    c = (np.arange(src_px) + 0.5) * mm_per_px
    s = 2.5 * d_mm
    dx = (c + s / 2) % s - s / 2
    r = np.hypot(dx[None, :], dx[:, None])
    edge = max(0.05 * d_mm, 0.5 * mm_per_px)
    disk = 0.5 * (1 + np.tanh((d_mm / 2 - r) / edge))
    img = bg + amp * disk + rng.normal(0, noise_sd, r.shape)
    return np.clip(img, 0, 255).astype(np.uint8), r


def score_print(z, d_mm, print_width_mm=PRINT_WIDTH_MM):
    """Measure a printed synthetic pattern: structure height and roughness."""
    h, w = z.shape
    px = print_width_mm / w
    c = (np.arange(w) + 0.5) * px
    s = 2.5 * d_mm
    dx = (c + s / 2) % s - s / 2
    cy = (np.arange(h) + 0.5) * px
    dy = (cy + s / 2) % s - s / 2
    r = np.hypot(dx[None, :], dy[:, None])
    centre = r <= max(0.2 * d_mm, 0.5 * px)
    backg = r >= 0.9 * d_mm
    mod = float(np.median(z[centre]) - np.median(z[backg]))
    rough = float(np.std(z[backg]))
    return mod, rough


def classify(d_mm, mod, rough):
    snr = mod / max(rough, 1e-3)
    if mod < MIN_RELIEF_MM or snr < SNR_MIN:
        return "lost"
    if d_mm < MIN_FEATURE_MM:
        return "too small to feel"
    if snr < SNR_OK or rough > LAYER_MM:
        return "rough"
    return "ok"


CLASSES = ["ok", "rough", "too small to feel", "lost"]


def build_map(sizes_mm, cnrs, resolution, blur, print_width_mm=PRINT_WIDTH_MM):
    """Run every synthetic pattern through TAMP-OS + print simulation.
    Patterns are generated directly at the preset resolution, so `cnr`
    (structure contrast / pixel noise) is the contrast-to-noise ratio the
    pipeline actually sees. Larger source images average noise down before
    this point, which `cnr_print()` accounts for when checking a real image."""
    out = []
    amp = 100.0
    for cnr in cnrs:
        for d in sizes_mm:
            img, _ = synth_bumps(d, amp / cnr, print_width_mm, resolution, amp=amp)
            hm = tamp_heightmap(img, resolution, blur)
            z = simulate_print(hm, print_width_mm)
            mod, rough = score_print(z, d, print_width_mm)
            out.append(dict(d_mm=d, cnr=cnr, mod=mod, rough=rough,
                            cls=classify(d, mod, rough), resolution=resolution,
                            blur=blur))
    return out

**Checking a real image + recommendations**

In [ ]:
# 6. Check + recommendations
def printed_noise_mm(img, resolution, blur, print_width_mm=PRINT_WIDTH_MM,
                     relief_mm=RELIEF_MM, seed=0):
    """How much grain (std, mm) the image's pixel noise leaves on the print:
    a noise field with the image's measured noise level goes through the same
    resampling, contrast stretch and blur as the image itself."""
    pil = to_pil(img).convert("L")
    g = np.array(pil, np.float32)
    sig = noise_sigma(g)
    arr = resample_gray(pil, resolution)
    lo, hi = np.percentile(arr, (2.0, 98.0))
    n = np.random.default_rng(seed).normal(0, sig, g.shape).astype(np.float32)
    w, h = pil.size
    n = np.array(Image.fromarray(n, mode="F").resize(
        (resolution, max(2, round(resolution * h / w))), Image.LANCZOS), np.float32)
    hm = n / (hi - lo + 1e-8)
    if blur > 0:
        hm = gaussian_filter(hm, blur)
    px_mm = print_width_mm / resolution
    z = gaussian_filter(hm * relief_mm, (NOZZLE_MM / 2.355) / px_mm)
    return float(np.std(z))


def recommend_settings(img, d_mm, print_width_mm=PRINT_WIDTH_MM,
                       blurs=(0.8, 1.2, 2.0, 3.0, 4.0, 6.0)):
    """For each preset x blur: does a structure of the image's size survive
    (synthetic test), and how much grain does the image's own noise leave
    (direct measurement)? Returns the best combinations, preferring Medium and
    its default blur, then more blur."""
    presets = calculate_presets(print_width_mm)
    order = ["Medium", "High", "Low"]
    trials = []
    for name in order:
        res = presets[name]["resolution"]
        clean, _ = synth_bumps(max(d_mm, 0.05), 0.5, print_width_mm, res)
        for b in blurs:
            z = simulate_print(tamp_heightmap(clean, res, b), print_width_mm)
            mod, _ = score_print(z, d_mm, print_width_mm)
            rough = printed_noise_mm(img, res, b, print_width_mm)
            trials.append(dict(preset=name, resolution=res, blur=b, mod=mod, rough=rough,
                               cls=classify(d_mm, mod, rough)))
    best_rank = min(CLASSES.index(t["cls"]) for t in trials)
    good = [t for t in trials if CLASSES.index(t["cls"]) == best_rank]
    def pref(t):
        dflt = presets[t["preset"]]["blur"]
        return (order.index(t["preset"]), 0 if t["blur"] == dflt else
                (1 if t["blur"] > dflt else 2), abs(t["blur"] - dflt))
    good.sort(key=pref)
    return good, trials


def check_image(img, print_width_mm=PRINT_WIDTH_MM, nozzle_mm=NOZZLE_MM, verbose=True):
    """Measure an image, place it on the map, and list problems + fixes."""
    pil = to_pil(img)
    gray_full = np.array(pil.convert("L"), np.float32)
    h, w = gray_full.shape
    rep = {"width": w, "height": h}
    issues = []

    # -- annotations first: they bias every other measurement
    boxes = find_annotations(gray_full)
    rep["annotations"] = boxes
    work = pil
    if boxes:
        work, keep = crop_out_annotations(pil, boxes)
        rep["crop_suggestion"] = keep
        issues.append(dict(
            issue="Embedded label / scale bar / data bar",
            detail=f"{len(boxes)} flat saturated box(es), e.g. at x={boxes[0][0]}-{boxes[0][2]}, "
                   f"y={boxes[0][1]}-{boxes[0][3]} px. They would print as raised plateaus.",
            tamp="No parameter removes them - crop before.",
            fiji=f"Rectangle tool, select x={keep[0]}-{keep[2]}, y={keep[1]}-{keep[3]} -> Image > Crop.",
            python=f"img, box = crop_out_annotations(img, {boxes})"))
    gray = np.array(work.convert("L"), np.float32)
    ww = gray.shape[1]

    # -- colour carries the contrast?
    rgb = np.array(work.convert("RGB"), np.float32)
    is_colour = float(np.mean(np.std(rgb, axis=2))) > 4
    if is_colour:
        gain, best = colour_gain(work)
        rep["colour_gain"], rep["best_channel"] = gain, best
        if gain > 1.5:
            fiji = ("Image > Color > Colour Deconvolution (vectors 'H&E' or 'H DAB'), keep the stain you need."
                    if best in ("hematoxylin", "DAB") else
                    f"Image > Color > Split Channels, keep the {best} channel.")
            issues.append(dict(
                issue="Contrast lives in the colour, not the brightness",
                detail=f"The {best} channel has {gain:.1f}x the contrast-to-noise of the grayscale "
                       "TAMP-OS uses; converting to gray throws part of the structure away.",
                tamp="No colour option in TAMP-OS - export a single channel first.",
                fiji=fiji, python=f"img = pick_channel(img, '{best}')"))

    # -- structure size on the print
    detr = gray - gaussian_filter(gray, ww / 6)   # ignore frame-scale trends / masks
    d_px, _ = feature_size_px(detr)
    d_mm = d_px / ww * print_width_mm
    rep["feature_mm"] = d_mm
    rep["touchable_fraction"] = touchable_fraction(detr, print_width_mm)
    if d_mm < MIN_FEATURE_MM or rep["touchable_fraction"] < 0.5:
        f = max(2.0 / d_mm, 1.5)
        issues.append(dict(
            issue="Structures are small on the print",
            detail=f"Dominant structure ~{d_mm:.1f} mm wide at {print_width_mm:.0f} mm print width "
                   f"(touchable from ~{MIN_FEATURE_MM} mm). Only {100*rep['touchable_fraction']:.0f}% "
                   "of the detail is at a touchable size.",
            tamp=f"Print wider: ~{print_width_mm*f:.0f} mm (print width / size_x).",
            fiji=f"Or crop to a region of interest ~1/{f:.1f} of the width "
                 f"(~{ww/f:.0f} px wide) -> Image > Crop. Each 2x crop doubles structure size.",
            python=f"img = crop(img, (x0, y0, x0 + {int(ww/f)}, y0 + {int(gray.shape[0]/f)}))"))

    # -- noise per preset resolution
    presets = calculate_presets(print_width_mm, nozzle_mm)
    cnr_by_res = {p["resolution"]: cnr_print(work, p["resolution"]) for p in presets.values()}
    rep["cnr"] = cnr_by_res

    # -- illumination / background trend
    r2 = illumination_trend(gray)
    rep["trend_r2"] = r2
    if r2 > 0.35:
        rad = int(min(max(3 * d_px, 20), ww / 3))
        issues.append(dict(
            issue="Uneven brightness across the frame",
            detail=f"A smooth brightness trend explains {100*r2:.0f}% of the image variance "
                   "(uneven illumination, vignetting, or one very large object - check by eye). "
                   "It prints as a tilted or domed base.",
            tamp="No parameter fixes it - flatten before.",
            fiji=f"Process > Subtract Background, rolling ball radius ~{rad} px "
                 "(larger than the biggest structure; tick 'Light background' if needed).",
            python=f"img = subtract_background(img, {rad})"))

    # -- polarity
    inv, bg_level = suggest_invert(gray, is_colour)
    rep["invert"], rep["background_level"] = inv, bg_level
    if inv:
        issues.append(dict(
            issue="Bright background",
            detail=f"Background is near white ({bg_level:.0f}/255), as in brightfield/histology. TAMP-OS "
                   "raises white, so the background would print high and the tissue sunk. "
                   "Check: are the structures you want to feel darker than the background?",
            tamp="Turn 'Invert relief' ON.", fiji="-", python="invert=True"))

    # -- small source
    if ww < presets["Medium"]["resolution"]:
        issues.append(dict(
            issue="Small source image",
            detail=f"{ww} px wide is below the Medium resolution ({presets['Medium']['resolution']} px): "
                   "TAMP-OS upsamples, which adds no detail (it still works).",
            tamp=f"Set resolution ~{ww} px (or Low) to keep files small; nothing is lost.",
            fiji="-", python="-"))

    # -- settings via simulation
    good, trials = recommend_settings(work, d_mm, print_width_mm)
    dflt = [t for t in trials if t["preset"] == "Medium"
            and t["blur"] == presets["Medium"]["blur"]][0]
    rep["default_outcome"], rep["default_grain_mm"] = dflt["cls"], dflt["rough"]
    rep["settings"] = good[0]
    rep["outcome"] = good[0]["cls"]
    pre = presets[good[0]["preset"]]
    if dflt["rough"] > LAYER_MM:
        g0 = good[0]
        fix = (f"Use {g0['preset']} with blur {g0['blur']} (simulated grain {g0['rough']:.2f} mm)."
               if g0["rough"] <= LAYER_MM else
               "No preset/blur combination removes the grain without erasing the structure - denoise before.")
        issues.append(dict(
            issue="Noisy / grainy",
            detail=f"With Medium defaults the pixel noise leaves ~{dflt['rough']:.2f} mm of grain on the print "
                   f"(more than one {LAYER_MM} mm layer) - it feels like sandpaper over the structure.",
            tamp=fix,
            fiji="Process > Filters > Median (radius 2) keeps edges sharper than blur; "
                 "if you can re-acquire, slower scan or frame averaging.",
            python="img = median_denoise(img, 5)"))
    rep["issues"] = issues
    if verbose:
        print_report(rep)
    return rep


def print_report(rep):
    s = rep["settings"]
    print(f"Image {rep['width']}x{rep['height']} px | dominant structure ~{rep['feature_mm']:.1f} mm on print "
          f"| contrast/noise (Medium) {list(rep['cnr'].values())[1]:.1f}")
    print(f"Predicted outcome with best settings: {rep['outcome'].upper()}  ->  "
          f"{s['preset']} ({s['resolution']} px), blur {s['blur']}, invert {'ON' if rep['invert'] else 'OFF'}")
    if not rep["issues"]:
        print("No problems found.")
    for i, it in enumerate(rep["issues"], 1):
        print(f"\n{i}. {it['issue']}\n   {it['detail']}\n   TAMP-OS: {it['tamp']}")
        if it["fiji"] != "-":
            print(f"   Fiji/ImageJ: {it['fiji']}")
        if it["python"] != "-":
            print(f"   Python (this notebook): {it['python']}")

**Figures**

In [ ]:
# ── Figures: relief preview, your image on the map, reference map, gallery ──
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, LightSource
from matplotlib.patches import Patch

STATUS = {"ok": "#0ca30c", "rough": "#fab219", "too small to feel": "#ec835a", "lost": "#d03b3b"}
SURF, INK, INK2 = "#fcfcfb", "#0b0b0b", "#52514e"
LABEL = {"ok": "OK", "rough": "grainy", "too small to feel": "too small\nto feel", "lost": "lost"}
plt.rcParams.update({"font.size": 9, "text.color": INK, "axes.labelcolor": INK2,
                     "xtick.color": INK2, "ytick.color": INK2})


def render_relief(z, ax, print_width_mm=PRINT_WIDTH_MM, title=None):
    """Hill-shaded preview of a printed relief (mm). Display only: no flip."""
    ls = LightSource(azdeg=315, altdeg=40)
    px = print_width_mm / z.shape[1]
    ax.imshow(ls.shade(z, cmap=plt.cm.gray, vert_exag=0.6 / px, blend_mode="hsv",
                       vmin=-1, vmax=RELIEF_MM + 1), interpolation="bilinear")
    ax.set_xticks([]); ax.set_yticks([])
    if title:
        ax.set_title(title, fontsize=9, color=INK2)


def print_preview(img, resolution=256, blur=1.2, invert=False, print_width_mm=PRINT_WIDTH_MM):
    return simulate_print(tamp_heightmap(img, resolution, blur, invert), print_width_mm)


def _map_panel(ax, resolution, blur, print_width_mm, sizes, cnrs, title):
    m = build_map(sizes, cnrs, resolution, blur, print_width_mm)
    g = np.array([CLASSES.index(r["cls"]) for r in m]).reshape(len(cnrs), len(sizes))
    cmap = ListedColormap([STATUS[c] for c in CLASSES])
    ax.set_facecolor(SURF)
    ax.pcolormesh(sizes, cnrs, g, cmap=cmap, vmin=-0.5, vmax=3.5, shading="nearest",
                  edgecolors=SURF, linewidth=1.0)
    ax.set_xscale("log"); ax.set_yscale("log")
    for sp in ax.spines.values():
        sp.set_visible(False)
    ax.axvline(MIN_FEATURE_MM, color=INK, lw=1, ls=(0, (3, 3)))
    ax.text(MIN_FEATURE_MM * 0.95, cnrs[0] * 1.15, f"{MIN_FEATURE_MM} mm = braille dot",
            fontsize=7.5, color=INK, va="bottom", ha="right", rotation=90,
            bbox=dict(boxstyle="square,pad=0.15", fc=SURF, ec="none", alpha=0.85))
    ax.set_title(title, fontsize=10, color=INK, loc="left")
    ax.set_xlabel("structure diameter on the print (mm)")
    ax.set_xticks([0.5, 1, 2, 5, 10]); ax.set_xticklabels(["0.5", "1", "2", "5", "10"])
    ax.set_yticks([0.5, 1, 2, 5, 10, 20, 50]); ax.set_yticklabels(["0.5", "1", "2", "5", "10", "20", "50"])
    for cls in CLASSES:                       # direct labels: never colour alone
        ii, jj = np.where(g == CLASSES.index(cls))
        if len(ii) > 6:
            k = np.argsort((ii - ii.mean()) ** 2 + (jj - jj.mean()) ** 2)[0]
            ax.text(sizes[jj[k]], cnrs[ii[k]], LABEL[cls], ha="center", va="center", fontsize=8,
                    fontweight="bold",
                    color=INK if cls in ("rough", "too small to feel") else "#ffffff")


def _legend(fig):
    handles = [Patch(color=STATUS[c], label=l) for c, l in
               [("ok", "OK: structure survives, smooth"),
                ("rough", "grainy: survives, but noise grain > 1 layer"),
                ("too small to feel", "printed, but smaller than a braille dot"),
                ("lost", "lost: < 0.5 mm relief left, or buried in noise")]]
    fig.legend(handles=handles, loc="lower center", ncol=4, frameon=False, fontsize=9,
               bbox_to_anchor=(0.5, -0.01))


def map_figure(out=None, print_width_mm=PRINT_WIDTH_MM, nozzle_mm=NOZZLE_MM):
    """Reference map: Low / Medium / High presets, plus Medium with more blur."""
    P = calculate_presets(print_width_mm, nozzle_mm)
    panels = [("Low", P["Low"]["blur"]), ("Medium", P["Medium"]["blur"]),
              ("High", P["High"]["blur"]), ("Medium", 3.0)]
    sizes = np.geomspace(0.3, 12, 22); cnrs = np.geomspace(0.5, 60, 18)
    fig, axs = plt.subplots(1, 4, figsize=(15, 4.6), sharey=True, facecolor=SURF)
    for ax, (name, blur) in zip(axs, panels):
        res = P[name]["resolution"]
        extra = "  (more blur)" if blur > P[name]["blur"] else ""
        _map_panel(ax, res, blur, print_width_mm, sizes, cnrs,
                   f"{name}: {res} px, blur {blur}{extra}\n1 px = {print_width_mm/res:.2f} mm on the print")
    axs[0].set_ylabel("contrast-to-noise at print resolution")
    _legend(fig)
    fig.suptitle(f"Will my image work?  Simulated TAMP-OS output: {print_width_mm:.0f} mm print, "
                 f"{nozzle_mm} mm nozzle, {LAYER_MM} mm layers, {RELIEF_MM} mm relief",
                 x=0.01, ha="left", fontsize=12, color=INK, fontweight="bold")
    fig.tight_layout(rect=(0, 0.07, 1, 0.97))
    if out:
        fig.savefig(out, dpi=200, facecolor=SURF)
    plt.show()


def show_check(img, rep, print_width_mm=PRINT_WIDTH_MM):
    """Original | defaults | recommended settings | where the image sits on the map."""
    s = rep["settings"]
    P = calculate_presets(print_width_mm)
    fig, axs = plt.subplots(1, 4, figsize=(15, 3.9), facecolor=SURF,
                            gridspec_kw=dict(width_ratios=[1, 1, 1, 1.3]))
    axs[0].imshow(to_pil(img).convert("RGB")); axs[0].set_xticks([]); axs[0].set_yticks([])
    axs[0].set_title("your image", fontsize=9, color=INK2)
    for b in rep["annotations"]:
        axs[0].add_patch(plt.Rectangle((b[0], b[1]), b[2] - b[0], b[3] - b[1],
                                       fill=False, ec=STATUS["lost"], lw=2))
    m = P["Medium"]
    render_relief(print_preview(img, m["resolution"], m["blur"], False, print_width_mm), axs[1],
                  print_width_mm, "Medium defaults (invert OFF)")
    render_relief(print_preview(img, s["resolution"], s["blur"], rep["invert"], print_width_mm), axs[2],
                  print_width_mm, f"suggested: {s['preset']}, blur {s['blur']}, "
                                  f"invert {'ON' if rep['invert'] else 'OFF'}")
    sizes = np.geomspace(0.3, 12, 16); cnrs = np.geomspace(0.5, 60, 12)
    _map_panel(axs[3], s["resolution"], s["blur"], print_width_mm, sizes, cnrs,
               f"your image on the map ({s['preset']}, blur {s['blur']})")
    axs[3].set_ylabel("contrast-to-noise")
    x = np.clip(rep["feature_mm"], 0.32, 11.5); y = np.clip(rep["cnr"][s["resolution"]], 0.52, 58)
    axs[3].plot(x, y, "*", ms=18, mfc=INK, mec=SURF, mew=1.5)
    left = x > 4
    axs[3].annotate("your image", (x, y), xytext=(-10 if left else 10, -12), textcoords="offset points",
                    ha="right" if left else "left", fontsize=8.5, fontweight="bold", color=INK,
                    bbox=dict(boxstyle="round,pad=0.2", fc=SURF, ec="none", alpha=0.9))
    fig.tight_layout()
    plt.show()
    print("Note: these previews are not flipped (display only). The GUI flips the STL itself.")


GALLERY = [
    # key, title, file, credit, fix description, function returning (fixed_image, settings)
    ("A", "Scale bar inside the image", "../examples/fept_spheres/SEM_5um_raw.png",
     "FePt spheres, SEM (this repo)",
     "Crop it out.\nFiji: rectangle → Image › Crop\nPython: crop_out_annotations()",
     lambda im: (crop_out_annotations(im, find_annotations(np.array(im.convert("L"), np.float32)))[0], {})),
    ("B", "Bright background (brightfield)", "examples/ihc_colon_no_known_restrictions.png",
     "IHC colon, CMMI via scikit-image\n(no known copyright restrictions)",
     "Invert relief ON (TAMP-OS), so the\ntissue is raised and the glass is low.",
     lambda im: (im, {"invert": True})),
    ("C", "Dark mask + uneven illumination", "examples/retina_fundus_cc0.jpg",
     "Fundus, M. Häggström (CC0)",
     "Crop inside the field of view, flatten\n(Fiji: Process › Subtract Background,\nrolling ball ~60 px), and Invert ON so\nthe dark vessels print raised.",
     lambda im: (subtract_background(crop(im, (int(im.width * .3), int(im.height * .3),
                                               int(im.width * .7), int(im.height * .7))), 60),
                 {"invert": True})),
    ("D", "Contrast only in the colour", "examples/synthetic_colour_only.png",
     "synthetic: red disks, equal brightness",
     "Export one channel.\nFiji: Image › Color › Split Channels\n(H&E/IHC: Colour Deconvolution)",
     lambda im: (pick_channel(im, "red"), {})),
    ("E", "Noisy (small, fast-scan image)", "examples/sem_crop_simulated_noise.png",
     "crop of A + simulated noise",
     "Blur 1.2 → 2.0 in TAMP-OS\n(the checker's suggestion).\nFiji alt.: Process › Filters › Median",
     lambda im: (im, {"blur": 2.0})),
    ("F", "Structures too small for 100 mm", "examples/hubble_xdf_public_domain.jpg",
     "Hubble XDF, NASA (public domain)",
     "Crop to 1/2 (or print 200 mm wide):\nevery 2× crop doubles structure size.",
     lambda im: (crop(im, (0, 0, im.width // 2, im.height // 2)), {})),
    ("G", "Tiny source image (102 px)", "examples/retina_detail_102px_cc0.png",
     "Retina detail, HRF database (CC0)",
     "Nothing to fix: TAMP-OS upsamples\nsmoothly. Pixel count is not the\nlimit, structure size on the print is.",
     None),
]


def gallery_figure(out=None, cases=GALLERY):
    fig, axs = plt.subplots(len(cases), 4, figsize=(13, 2.55 * len(cases)), facecolor=SURF,
                            gridspec_kw=dict(width_ratios=[1, 1, 1, 1.25]))
    for (key, title, path, credit, fix, fn), row in zip(cases, axs):
        im = to_pil(path)
        rep = check_image(im, verbose=False)
        row[0].imshow(im.convert("RGB")); row[0].set_xticks([]); row[0].set_yticks([])
        row[0].set_title(f"{key}  {title}", loc="left", fontsize=10, color=INK, fontweight="bold")
        row[0].set_xlabel(credit, fontsize=7, color=INK2)
        render_relief(print_preview(im), row[1], title="TAMP-OS defaults")
        if fn is None:
            row[2].axis("off")
        else:
            fixed, kw = fn(im)
            render_relief(print_preview(fixed, **kw), row[2], title="after the fix")
        row[3].axis("off")
        issues = [i["issue"] for i in rep["issues"]] or ["no problems found"]
        row[3].text(0, 0.98, "Checker found:\n" + "\n".join("• " + i for i in issues) + "\n\nFix:\n" + fix,
                    va="top", fontsize=8.3, color=INK, transform=row[3].transAxes, linespacing=1.35)
    fig.tight_layout(h_pad=1.2)
    if out:
        fig.savefig(out, dpi=160, facecolor=SURF)
    plt.show()
print("All functions loaded OK.")

### Step 4 — Check your image

Set the path to your image and run the cell. The report lists each problem with three ways to fix it: a **TAMP-OS** setting, an **ImageJ/Fiji** step, or a **Python** line for Step 5.

The preview shows your image, the relief with Medium defaults, the relief with the suggested settings, and where your image sits on the map.

In [ ]:
IMAGE_PATH = "../examples/fept_spheres/SEM_5um_raw.png"   # <- your image
PRINT_WIDTH = 100.0                                    # mm, same as in the GUI

img = to_pil(IMAGE_PATH)
rep = check_image(img, print_width_mm=PRINT_WIDTH)
show_check(img, rep, print_width_mm=PRINT_WIDTH)

### Step 5 — Apply a fix and check again *(optional)*

Copy the Python line(s) from the report, run them, and check again. Then save the fixed image and load it in the GUI. Example for the SEM image above (cropping out the scale bar):

In [ ]:
fixed, box = crop_out_annotations(img, rep["annotations"])    # from the report
# other fixes, if the report suggests them:
# fixed = pick_channel(fixed, "hematoxylin")    # colour -> one channel / stain
# fixed = subtract_background(fixed, 60)        # flatten uneven illumination
# fixed = crop(fixed, (x0, y0, x1, y1))         # region of interest

rep2 = check_image(fixed, print_width_mm=PRINT_WIDTH)
show_check(fixed, rep2, print_width_mm=PRINT_WIDTH)

out = Path(IMAGE_PATH).with_name(Path(IMAGE_PATH).stem + "_fixed.png")
fixed.save(out)
print(f"Saved {out} - load this file in tamp_batch_gui_v2.ipynb")

### Step 6 — Reference map for your printer *(optional)*

The map shows, for each preset, which combinations of **structure size on the print** and **contrast-to-noise** survive printing. Change the printer settings in Step 3 and run again to make the map for your machine (~20 s).

How to read it:
- **Left of the dashed line** (1.5 mm, a braille dot), structures print but are probably too small to tell apart by touch. **Fix: crop or print wider.**
- **Red on the left**: the preset's resolution and blur erase the structure entirely.
- **Yellow at the bottom**: the structure survives but noise leaves grain thicker than one layer. **Fix: more blur.** Compare the 2nd and 4th panels: more blur removes the grain, at the cost of the smallest structures.

In [ ]:
map_figure("assets/image_map.png", print_width_mm=PRINT_WIDTH_MM, nozzle_mm=NOZZLE_MM)

### Step 7 — Example gallery *(optional)*

Real, copyright-free images (licences in `examples/LICENSES.md`), each showing one typical problem: original → TAMP-OS defaults → after the fix.

In [ ]:
gallery_figure("assets/image_gallery.png")

### Notes and limitations

- **Pixel count is rarely the problem.** TAMP-OS resizes every image to the preset resolution (256 px for a 0.4 mm nozzle at 100 mm). What matters is how large the structures end up **in mm on the print**. Large noisy images also get some denoising for free: shrinking 1024 → 256 px averages the noise by ~4×.
- **The Low preset is heavily smoothed**: 128 px with blur 2 ≈ 1.6 mm of blur on a 100 mm print. It erases structures below ~3 mm, so use it for drafts only.
- **Brightness is not height.** SEM edge glow, shadows and shading in photographs of 3D objects become false relief. The checker cannot detect this, so judge by eye.
- **Polarity** is a suggestion. Ask yourself: *are the structures I want to feel darker than the background?* If yes, invert ON.
- **The structure-size estimate assumes uncorrelated noise.** After a median or denoising filter it can underestimate the size, so trust the preview.
- The thresholds (1.5 mm, 0.5 mm, grain > 1 layer) are starting points. Change them in Step 3 once user testing gives better numbers.